# 03 — MinHash and Locality-Sensitive Hashing

In this notebook, we identify games with similar player audiences.

Each game is represented by the set of users who have played it for at least 2 hours. We first explore exact Jaccard similarity and then implement MinHash and Locality-Sensitive Hashing (LSH) to find similar games efficiently.

In [1]:
from pathlib import Path

import pandas as pd


DATA_PATH = Path("../data/processed/interactions_filtered.parquet")

interactions = pd.read_parquet(DATA_PATH)

print("Interactions:", len(interactions))
print("Users:", interactions["user_id"].nunique())
print("Games:", interactions["game_id"].nunique())

interactions.head()

Interactions: 1871351
Users: 66754
Games: 2402


,user_id,game_id,game_name,playtime_forever,playtime_2weeks
0,wolop,4000,Garry's Mod,642773,7299
1,Evilutional,212200,Mabinogi,635295,0
2,76561198019826668,42710,Call of Duty: Black Ops - Multiplayer,632452,8126
3,76561198039832932,4000,Garry's Mod,613411,588
4,tsunamitad,72200,Universe Sandbox,600068,0


## Represent each game as a set of users

To compare games based on player behaviour, we represent each game by the set of users who are engaged with it.

For example:

- Game A → {User 1, User 2, User 3}
- Game B → {User 2, User 3, User 4}

These sets can then be compared using Jaccard similarity.

In [2]:
game_users = (
    interactions
    .groupby("game_id")["user_id"]
    .apply(set)
    .to_dict()
)

print("Number of games:", len(game_users))

# Show the number of engaged users for a few games.
for game_id, users in list(game_users.items())[:5]:
    print(game_id, "->", len(users), "users")

Number of games: 2402
10 -> 2511 users
100 -> 517 users
10090 -> 3036 users
10130 -> 78 users
10150 -> 771 users


## Exact Jaccard similarity

Jaccard similarity measures how similar two sets are.

For two games, we compare their sets of engaged users:

\[
J(A, B) = \frac{|A \cap B|}{|A \cup B|}
\]

A Jaccard similarity of:

- `0` means the games share no engaged users
- `1` means they have exactly the same engaged users
- values in between indicate partial overlap

In [3]:
def jaccard_similarity(users_a, users_b):
    """Calculate the Jaccard similarity between two sets of users."""
    intersection = users_a & users_b
    union = users_a | users_b

    return len(intersection) / len(union)


game_ids = list(game_users.keys())

game_a = game_ids[0]
game_b = game_ids[1]

similarity = jaccard_similarity(
    game_users[game_a],
    game_users[game_b]
)

print("Game A:", game_a)
print("Game B:", game_b)
print("Jaccard similarity:", similarity)

Game A: 10
Game B: 100
Jaccard similarity: 0.14393653192293163


In [4]:
users_a = game_users[game_a]
users_b = game_users[game_b]

intersection = users_a & users_b
union = users_a | users_b

print("Users in Game A:", len(users_a))
print("Users in Game B:", len(users_b))
print("Shared users:", len(intersection))
print("Users in union:", len(union))
print("Jaccard similarity:", len(intersection) / len(union))

Users in Game A: 2511
Users in Game B: 517
Shared users: 381
Users in union: 2647
Jaccard similarity: 0.14393653192293163


In [5]:
def compare_games(game_a, game_b):
    """Print overlap statistics for two games."""
    users_a = game_users[game_a]
    users_b = game_users[game_b]

    intersection = users_a & users_b
    union = users_a | users_b

    similarity = len(intersection) / len(union)

    print("Game A:", game_a)
    print("Game B:", game_b)
    print("Users in Game A:", len(users_a))
    print("Users in Game B:", len(users_b))
    print("Shared users:", len(intersection))
    print("Jaccard similarity:", similarity)

In [14]:
compare_games("10", "80")

Game A: 10
Game B: 80
Users in Game A: 2511
Users in Game B: 1149
Shared users: 750
Jaccard similarity: 0.25773195876288657


## Why not compare every game pair directly?

Exact Jaccard similarity works well for comparing individual games, but comparing every possible pair becomes expensive.

With 2,402 games, there are almost 2.9 million unique game pairs.

MinHash gives us a compact signature for each game's user set that approximately preserves Jaccard similarity. LSH can then use these signatures to quickly identify promising candidate pairs.

The overall idea is:

1. Convert each game into a user set
2. Create a MinHash signature for each game
3. Use LSH to find likely similar game pairs
4. Compute exact Jaccard similarity only for those candidates

In [15]:
number_of_games = len(game_users)

number_of_pairs = (
    number_of_games * (number_of_games - 1)
) // 2

print("Number of games:", number_of_games)
print("Possible game pairs:", number_of_pairs)

Number of games: 2402
Possible game pairs: 2883601


In [16]:
import random
import time


def jaccard_similarity_fast(users_a, users_b):
    """Calculate Jaccard similarity without explicitly building the union."""
    intersection_size = len(users_a & users_b)
    union_size = len(users_a) + len(users_b) - intersection_size

    return intersection_size / union_size


game_ids = list(game_users.keys())

# Generate 100,000 random game pairs for the benchmark.
number_of_test_pairs = 100_000

test_pairs = [
    random.sample(game_ids, 2)
    for _ in range(number_of_test_pairs)
]

start_time = time.perf_counter()

for game_a, game_b in test_pairs:
    jaccard_similarity_fast(
        game_users[game_a],
        game_users[game_b]
    )

elapsed_time = time.perf_counter() - start_time

comparisons_per_second = number_of_test_pairs / elapsed_time
estimated_full_time = number_of_pairs / comparisons_per_second

print(f"Test comparisons: {number_of_test_pairs:,}")
print(f"Benchmark time: {elapsed_time:.2f} seconds")
print(f"Comparisons per second: {comparisons_per_second:,.0f}")
print(f"Estimated time for all {number_of_pairs:,} pairs: {estimated_full_time:.2f} seconds")

Test comparisons: 100,000
Benchmark time: 0.69 seconds
Comparisons per second: 144,149
Estimated time for all 2,883,601 pairs: 20.00 seconds


### Benchmark interpretation

On the current filtered dataset, 100,000 exact Jaccard comparisons took approximately 0.69 seconds, corresponding to roughly 144,000 comparisons per second. This suggests that exhaustively comparing all 2.88 million game pairs would take around 20 seconds on the current machine.

The earlier filtering step also has a substantial computational impact. The original interaction dataset contains 10,978 unique games, corresponding to more than 60 million possible game pairs. After requiring at least 2 hours of playtime and at least 50 engaged players per game, only 2,402 games remain, reducing the number of possible comparisons to approximately 2.88 million. This is roughly a 21-fold reduction.

Therefore, exhaustive Jaccard comparison is feasible for the filtered dataset. However, this is partly a consequence of the filtering, and pairwise comparison still scales quadratically with the number of games.

MinHash and LSH therefore remain useful as a more scalable approach. Since exhaustive Jaccard is feasible at our current scale, we can also use it as a ground-truth baseline to evaluate how effectively MinHash and LSH identify similar game pairs.

## Exhaustive Jaccard baseline

Since exhaustive comparison is feasible for the filtered dataset, we calculate the exact Jaccard similarity for every pair of games.

This provides a ground-truth baseline that can later be used to evaluate the MinHash and LSH results.

In [17]:
import numpy as np
import time


game_ids = list(game_users.keys())
number_of_games = len(game_ids)
number_of_pairs = number_of_games * (number_of_games - 1) // 2

# Store all exact similarities efficiently in a NumPy array.
exact_jaccard_scores = np.empty(number_of_pairs, dtype=np.float32)

start_time = time.perf_counter()

index = 0

for i in range(number_of_games):
    users_a = game_users[game_ids[i]]

    for j in range(i + 1, number_of_games):
        users_b = game_users[game_ids[j]]

        intersection_size = len(users_a & users_b)
        union_size = len(users_a) + len(users_b) - intersection_size

        exact_jaccard_scores[index] = intersection_size / union_size
        index += 1

elapsed_time = time.perf_counter() - start_time

print(f"Comparisons completed: {index:,}")
print(f"Time taken: {elapsed_time:.2f} seconds")

Comparisons completed: 2,883,601
Time taken: 15.82 seconds


## Distribution of exact Jaccard similarities

Before defining which game pairs should be considered similar, we inspect the distribution of exact Jaccard scores across all game pairs.

In [18]:
print("Minimum:", exact_jaccard_scores.min())
print("Median:", np.median(exact_jaccard_scores))
print("Mean:", exact_jaccard_scores.mean())
print("90th percentile:", np.percentile(exact_jaccard_scores, 90))
print("95th percentile:", np.percentile(exact_jaccard_scores, 95))
print("99th percentile:", np.percentile(exact_jaccard_scores, 99))
print("Maximum:", exact_jaccard_scores.max())

Minimum: 0.0
Median: 0.0057471264
Mean: 0.015009959
90th percentile: 0.03550296
95th percentile: 0.0621442
99th percentile: 0.16
Maximum: 0.8939394


In [19]:
thresholds = [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]

for threshold in thresholds:
    count = (exact_jaccard_scores >= threshold).sum()
    percentage = count / len(exact_jaccard_scores) * 100

    print(
        f"Jaccard >= {threshold:.2f}: "
        f"{count:,} pairs ({percentage:.2f}%)"
    )

Jaccard >= 0.05: 192,617 pairs (6.68%)
Jaccard >= 0.10: 71,297 pairs (2.47%)
Jaccard >= 0.15: 33,372 pairs (1.16%)
Jaccard >= 0.20: 16,066 pairs (0.56%)
Jaccard >= 0.30: 3,742 pairs (0.13%)
Jaccard >= 0.50: 206 pairs (0.01%)


## MinHash intuition

Exact Jaccard similarity requires comparing the full user sets of two games.

MinHash instead creates a compact signature for each set. The key property is that two sets with high Jaccard similarity are likely to have similar MinHash signatures.

We start with a single hash function before extending this to a full signature.

## One MinHash function

To apply MinHash, we first map each user ID to an integer.

We then apply a hash function to every user in a game's user set and keep the smallest resulting hash value. This minimum value is the game's MinHash value for that hash function.

In [20]:
# Give every unique user a reproducible integer ID.
unique_users = sorted(interactions["user_id"].unique())

user_to_int = {
    user_id: index
    for index, user_id in enumerate(unique_users)
}

print("Number of users:", len(user_to_int))

Number of users: 66754


In [21]:
# Parameters for one simple hash function.
a = 37
b = 17
prime = 100_003


def hash_user(user_int):
    """Hash an integer user ID."""
    return (a * user_int + b) % prime


def minhash_value(users):
    """Return the minimum hash value for a set of users."""
    hashed_users = [
        hash_user(user_to_int[user])
        for user in users
    ]

    return min(hashed_users)


game_id = game_ids[0]

value = minhash_value(game_users[game_id])

print("Game:", game_id)
print("Number of users:", len(game_users[game_id]))
print("MinHash value:", value)

Game: 10
Number of users: 2511
MinHash value: 99


In [22]:
game_a = game_ids[0]
game_b = game_ids[1]

minhash_a = minhash_value(game_users[game_a])
minhash_b = minhash_value(game_users[game_b])

print("Game A:", game_a, "MinHash:", minhash_a)
print("Game B:", game_b, "MinHash:", minhash_b)
print("Same MinHash:", minhash_a == minhash_b)

Game A: 10 MinHash: 99
Game B: 100 MinHash: 546
Same MinHash: False


## MinHash signatures

A single MinHash value is too noisy to estimate similarity reliably.

Instead, we use many different hash functions. Each hash function produces one minimum value, and together these values form a MinHash signature.

The similarity between two signatures is estimated as the fraction of positions where the two signatures are equal.

In [23]:
import numpy as np


NUM_HASHES = 100
PRIME = 100_003

# Create reproducible random hash functions.
rng = np.random.default_rng(seed=42)

a_values = rng.integers(
    1,
    PRIME,
    size=NUM_HASHES
)

b_values = rng.integers(
    0,
    PRIME,
    size=NUM_HASHES
)


def create_minhash_signature(users):
    """Create a MinHash signature for a set of users."""
    user_integers = np.array([
        user_to_int[user]
        for user in users
    ])

    signature = []

    for a, b in zip(a_values, b_values):
        hashed_users = (a * user_integers + b) % PRIME
        signature.append(hashed_users.min())

    return np.array(signature)

In [24]:
signature_a = create_minhash_signature(game_users[game_a])
signature_b = create_minhash_signature(game_users[game_b])

estimated_similarity = np.mean(signature_a == signature_b)

exact_similarity = jaccard_similarity(
    game_users[game_a],
    game_users[game_b]
)

print("Exact Jaccard similarity:", exact_similarity)
print("MinHash estimate:", estimated_similarity)
print("Matching signature positions:", (signature_a == signature_b).sum())
print("Number of hash functions:", NUM_HASHES)

Exact Jaccard similarity: 0.14393653192293163
MinHash estimate: 0.08
Matching signature positions: 8
Number of hash functions: 100


In [25]:
for num_hashes in [50, 100, 200, 500]:
    rng = np.random.default_rng(seed=42)

    a_values_test = rng.integers(
        1,
        PRIME,
        size=num_hashes
    )

    b_values_test = rng.integers(
        0,
        PRIME,
        size=num_hashes
    )

    def create_signature_with_hashes(users):
        user_integers = np.array([
            user_to_int[user]
            for user in users
        ])

        signature = []

        for a, b in zip(a_values_test, b_values_test):
            hashed_users = (a * user_integers + b) % PRIME
            signature.append(hashed_users.min())

        return np.array(signature)

    signature_a = create_signature_with_hashes(game_users[game_a])
    signature_b = create_signature_with_hashes(game_users[game_b])

    estimate = np.mean(signature_a == signature_b)

    print(
        f"{num_hashes} hashes -> "
        f"estimated similarity: {estimate:.3f}"
    )

50 hashes -> estimated similarity: 0.140
100 hashes -> estimated similarity: 0.080
200 hashes -> estimated similarity: 0.165
500 hashes -> estimated similarity: 0.148
